> **Public reproducibility copy.** Notebook outputs and execution counts were cleared before public packaging to avoid local-path/account leakage. The scientific code and configuration cells are preserved. See the repository README for required data paths and reproducibility scope.

# AMD Transformer Comparison — v6 Full Experiment Clean

Versi penuh yang sudah membersihkan error post-processing `x.mode`, warning `torch.load`, mempertahankan kompatibilitas Windows, menggunakan `NUM_WORKERS=0`, `BATCH_SIZE=2`, lima seed, lima outer fold, serta dua mode fine-tuning.


In [ ]:
%pip install -q torch torchvision timm pandas numpy pillow scikit-learn


## Full experiment
Periksa seluruh path pada `CONFIG`. Jalankan smoke test terlebih dahulu dengan `SMOKE_TEST=True`, satu seed, dan satu mode.


In [ ]:
from pathlib import Path
import os, gc, json, time, random, hashlib, platform
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import timm
from timm.data import resolve_model_data_config
from timm.data.transforms_factory import create_transform

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

CONFIG = {
    "DATASET_ROOT": Path("AMD_DATASET_594_PRODUCTION_v1"),
    "ROI_SEARCH_ROOT": Path("AMD_DATASET_594_PRODUCTION_v1\ROI_594"),  # Set exact frozen-ROI folder here when known.
    "OUTER_FOLD_FILE": Path("AMD_DATASET_594_PRODUCTION_v1/outer_fold_assignments.csv"),
    "NESTED_PREDICTIONS_FILE": Path("AMD_NESTEDCV_MULTISEED_v1/nested_sample_level_predictions.csv"),
    "OUTPUT_ROOT": Path("AMD_TRANSFORMER_COMPARISON_v1"),
    "MODEL_NAME": "vit_base_patch16_224.augreg_in21k_ft_in1k",
    "TRAINING_MODES": ["PARTIAL_LAST_BLOCK", "FULL_FINE_TUNING"],
    "SEEDS": [42, 123, 2026, 3407, 7777],
    "VALIDATION_FRACTION": 0.15,
    "MAX_EPOCHS": 30,
    "PATIENCE": 6,
    "BATCH_SIZE": 2,
    "NUM_WORKERS": 0,
    "WEIGHT_DECAY": 1e-4,
    "LEARNING_RATE": {
        "PARTIAL_LAST_BLOCK": 3e-5,
        "FULL_FINE_TUNING": 1e-5,
    },
    "USE_CLASS_WEIGHTS": True,
    "USE_AMP": True,
    "SMOKE_TEST": False,
}

def seed_all(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

CONFIG["OUTPUT_ROOT"].mkdir(parents=True, exist_ok=True)
(CONFIG["OUTPUT_ROOT"]/"checkpoints").mkdir(exist_ok=True)

manifest_path = CONFIG["DATASET_ROOT"]/"provenance"/"release_manifest_594_sha256.csv"
manifest = pd.read_csv(manifest_path, dtype={"sample_id":str})
folds = pd.read_csv(CONFIG["OUTER_FOLD_FILE"], dtype={"sample_id":str})
manifest["sample_id"] = manifest["sample_id"].str.strip()
folds["sample_id"] = folds["sample_id"].str.strip()
if "outer_fold" in manifest.columns:
    manifest = manifest.drop(columns=["outer_fold"])
df = manifest.merge(folds[["sample_id","outer_fold"]], on="sample_id", validate="one_to_one")
if len(df) != 594 or df["outer_fold"].isna().any():
    raise ValueError("Production manifest/fold validation failed.")

print("FULL EXPERIMENT CLEAN VERSION v6 LOADED")

path_cols = [
    "roi_path", "image_path", "file_path", "relative_path", "path",
    "roi_file", "roi_filename", "image_file", "image_filename",
    "filename", "file_name", "basename"
]
path_col = next((c for c in path_cols if c in df.columns), None)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

def all_images(root):
    return [
        p for p in Path(root).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

def roi_keyword_score(path):
    text = " ".join(part.lower() for part in Path(path).parts)
    score = 0
    for word, weight in {
        "roi": 8, "crop": 6, "cropped": 6, "macula": 5,
        "production": 3, "preprocessed": 2, "processed": 2,
        "original": -8, "raw": -7, "thumbnail": -6,
        "preview": -5, "backup": -5
    }.items():
        if word in text:
            score += weight
    return score

def exact_stem_matches(images, sample_id):
    sid = str(sample_id).strip().lower()
    return [p for p in images if p.stem.lower() == sid]

def infer_roi_root(dataset_root, sample_ids):
    images = all_images(dataset_root)
    if not images:
        raise FileNotFoundError(f"No image files found under {dataset_root}")

    # Include each image's parent directory AND all ancestors up to
    # DATASET_ROOT. This is required when the 594 ROI images are split into
    # class folders such as ROI_594/NORMAL, ROI_594/AMD_DRY, ROI_594/AMD_WET.
    dataset_root = Path(dataset_root).resolve()
    candidate_root_set = set()

    for p in images:
        if p.stem.lower() not in sample_ids:
            continue

        current = p.parent.resolve()
        while True:
            candidate_root_set.add(current)
            if current == dataset_root:
                break
            if dataset_root not in current.parents:
                break
            current = current.parent

    candidate_roots = sorted(
        candidate_root_set,
        key=lambda p: (len(p.parts), str(p))
    )

    rows = []
    for root in candidate_roots:
        root_images = [p for p in images if p.parent == root or root in p.parents]
        unique = ambiguous = missing = 0
        for sid in sample_ids:
            n = len(exact_stem_matches(root_images, sid))
            if n == 1:
                unique += 1
            elif n > 1:
                ambiguous += 1
            else:
                missing += 1
        rows.append({
            "candidate_root": str(root),
            "unique_coverage": unique,
            "ambiguous_count": ambiguous,
            "missing_count": missing,
            "roi_keyword_score": roi_keyword_score(root),
            "depth": len(root.parts),
        })

    report = pd.DataFrame(rows).sort_values(
        ["unique_coverage", "ambiguous_count", "missing_count",
         "roi_keyword_score", "depth"],
        ascending=[False, True, True, False, False]
    )
    report.to_csv(
        CONFIG["OUTPUT_ROOT"] / "roi_root_autodetection_report.csv",
        index=False
    )

    viable = report[
        (report["unique_coverage"] == len(sample_ids)) &
        (report["ambiguous_count"] == 0) &
        (report["missing_count"] == 0)
    ].copy()

    if len(viable) == 0:
        print(report.head(20).to_string(index=False))
        raise RuntimeError(
            "No folder uniquely maps all 594 sample IDs to one image. "
            "Open roi_root_autodetection_report.csv and set "
            "CONFIG['ROI_SEARCH_ROOT'] to the exact frozen ROI folder."
        )

    best = viable.iloc[0]
    tied = viable[
        (viable["roi_keyword_score"] == best["roi_keyword_score"]) &
        (viable["depth"] == best["depth"])
    ]
    if len(tied) > 1:
        print(tied.to_string(index=False))
        raise RuntimeError(
            "Multiple equally plausible ROI folders remain. Set "
            "CONFIG['ROI_SEARCH_ROOT'] explicitly."
        )

    selected = Path(best["candidate_root"])
    print("Auto-detected frozen ROI root:", selected)
    return selected

explicit_root = CONFIG.get("ROI_SEARCH_ROOT")
if explicit_root:
    roi_root = Path(explicit_root)
    if not roi_root.exists():
        raise FileNotFoundError(f"ROI_SEARCH_ROOT does not exist: {roi_root}")
    print("Using configured frozen ROI root:", roi_root)
else:
    roi_root = infer_roi_root(
        CONFIG["DATASET_ROOT"],
        set(df["sample_id"].astype(str).str.strip().str.lower())
    )

roi_images = all_images(roi_root)
stem_index = {}
name_index = {}
for p in roi_images:
    stem_index.setdefault(p.stem.lower(), []).append(p)
    name_index.setdefault(p.name.lower(), []).append(p)

def resolve_manifest_value(raw_value):
    value = str(raw_value).strip()
    p = Path(value)
    if p.is_absolute() and p.exists():
        return p
    candidate = roi_root / p
    if candidate.exists():
        return candidate
    matches = name_index.get(p.name.lower(), [])
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            f"Ambiguous manifest reference '{value}' inside {roi_root}: "
            f"{len(matches)} files."
        )
    return Path("__MISSING__") / value

if path_col is not None:
    print("Using manifest image-reference column:", path_col)
    df["resolved_path"] = df[path_col].map(resolve_manifest_value)
else:
    print("No path/filename column found; resolving exact sample_id stems inside ROI root.")
    def resolve_sample(sample_id):
        matches = stem_index.get(str(sample_id).strip().lower(), [])
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(
                f"Selected ROI root is ambiguous for '{sample_id}': "
                f"{len(matches)} files."
            )
        return Path("__MISSING__") / str(sample_id)
    df["resolved_path"] = df["sample_id"].map(resolve_sample)

missing_cols = ["sample_id"] + ([path_col] if path_col is not None else [])
missing = df.loc[~df["resolved_path"].map(Path.exists), missing_cols]
if len(missing):
    raise FileNotFoundError(
        "Could not resolve one or more frozen ROI images. First rows:\n" +
        missing.head(10).to_string(index=False)
    )

if df["resolved_path"].astype(str).duplicated().any():
    dup = df.loc[
        df["resolved_path"].astype(str).duplicated(keep=False),
        ["sample_id", "resolved_path"]
    ]
    raise RuntimeError(
        "One ROI file was assigned to multiple samples:\n" +
        dup.head(20).to_string(index=False)
    )

print("Resolved frozen ROI images:", len(df))
print("Selected ROI root:", roi_root)

labels = sorted(df["class_label"].unique())
label_to_int = {x:i for i,x in enumerate(labels)}
int_to_label = {i:x for x,i in label_to_int.items()}

class ROIDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row["resolved_path"]) as im:
            image = im.convert("RGB")
        return {
            "image": self.transform(image),
            "target": torch.tensor(label_to_int[row["class_label"]], dtype=torch.long),
            "sample_id": row["sample_id"],
            "source_dataset": row["source_dataset"],
            "outer_fold": int(row["outer_fold"]),
        }

def make_model(mode):
    model = timm.create_model(CONFIG["MODEL_NAME"], pretrained=True, num_classes=len(labels))
    if mode == "PARTIAL_LAST_BLOCK":
        for p in model.parameters(): p.requires_grad = False
        for p in model.blocks[-1].parameters(): p.requires_grad = True
        for p in model.norm.parameters(): p.requires_grad = True
        for p in model.head.parameters(): p.requires_grad = True
    elif mode == "FULL_FINE_TUNING":
        for p in model.parameters(): p.requires_grad = True
    else:
        raise ValueError(mode)
    return model

def split_outer_train(frame, seed):
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=CONFIG["VALIDATION_FRACTION"], random_state=seed)
    tr, va = next(splitter.split(np.zeros(len(frame)), frame["class_label"]))
    return frame.iloc[tr].copy(), frame.iloc[va].copy()

def loader(frame, transform, shuffle, seed):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(
        ROIDataset(frame, transform), batch_size=CONFIG["BATCH_SIZE"],
        shuffle=shuffle, num_workers=CONFIG["NUM_WORKERS"],
        pin_memory=torch.cuda.is_available(), generator=g
    )

def class_weights(frame):
    counts = frame["class_label"].map(label_to_int).value_counts().sort_index().to_numpy(float)
    return torch.tensor(len(frame)/(len(labels)*counts), dtype=torch.float32)

def epoch_pass(model, dl, criterion, device, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    losses, yt, yp = [], [], []
    start = time.perf_counter()
    for b in dl:
        x = b["image"].to(device, non_blocking=True)
        y = b["target"].to(device, non_blocking=True)
        if training: optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, enabled=CONFIG["USE_AMP"] and device.type=="cuda"):
            logits = model(x); loss = criterion(logits,y)
        if training:
            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
        pred = logits.argmax(1)
        losses.append(float(loss.detach().cpu()))
        yt.extend(y.detach().cpu().tolist()); yp.extend(pred.detach().cpu().tolist())
    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(yt,yp),
        "macro_f1": f1_score(yt,yp,average="macro",zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(yt,yp),
        "seconds": time.perf_counter()-start,
    }

@torch.no_grad()
def predict(model, dl, device):
    model.eval(); rows=[]
    if device.type=="cuda": torch.cuda.synchronize()
    start=time.perf_counter()
    for b in dl:
        x=b["image"].to(device,non_blocking=True)
        with autocast(device_type=device.type, enabled=CONFIG["USE_AMP"] and device.type=="cuda"):
            prob=torch.softmax(model(x),dim=1)
        pred=prob.argmax(1).cpu().numpy(); prob=prob.cpu().numpy()
        for i,sid in enumerate(b["sample_id"]):
            row={"sample_id":sid,"source_dataset":b["source_dataset"][i],
                 "outer_fold":int(b["outer_fold"][i]),
                 "predicted_label":int_to_label[int(pred[i])]}
            for j,c in enumerate(labels): row[f"prob_{c}"]=float(prob[i,j])
            rows.append(row)
    if device.type=="cuda": torch.cuda.synchronize()
    return pd.DataFrame(rows), time.perf_counter()-start

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
print("Batch size:", CONFIG["BATCH_SIZE"])
print("DataLoader workers:", CONFIG["NUM_WORKERS"])
epochs_out=[]; folds_out=[]; preds_out=[]; ckpt_out=[]

for seed in CONFIG["SEEDS"]:
    seed_all(seed)
    for mode in CONFIG["TRAINING_MODES"]:
        for fold in range(1,6):
            print(f"START seed={seed} mode={mode} fold={fold}", flush=True)
            outer_train=df[df.outer_fold!=fold].copy()
            outer_test=df[df.outer_fold==fold].copy()
            train_df,val_df=split_outer_train(outer_train,seed+fold)
            if CONFIG["SMOKE_TEST"]:
                train_df=train_df.groupby("class_label",group_keys=False).head(12)
                val_df=val_df.groupby("class_label",group_keys=False).head(4)
            model=make_model(mode)
            cfg=resolve_model_data_config(model)
            train_tf=create_transform(**cfg,is_training=True,hflip=0.5,color_jitter=0.15,auto_augment=None,re_prob=0)
            eval_tf=create_transform(**cfg,is_training=False)
            train_dl=loader(train_df,train_tf,True,seed+fold)
            val_dl=loader(val_df,eval_tf,False,seed+fold)
            test_dl=loader(outer_test,eval_tf,False,seed+fold)
            model=model.to(device)
            weights=class_weights(train_df).to(device) if CONFIG["USE_CLASS_WEIGHTS"] else None
            criterion=nn.CrossEntropyLoss(weight=weights)
            optimizer=torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=CONFIG["LEARNING_RATE"][mode],weight_decay=CONFIG["WEIGHT_DECAY"])
            scaler=GradScaler("cuda") if CONFIG["USE_AMP"] and device.type=="cuda" else None
            max_epochs=2 if CONFIG["SMOKE_TEST"] else CONFIG["MAX_EPOCHS"]
            best=-1; best_epoch=0; stale=0
            ckpt=CONFIG["OUTPUT_ROOT"]/"checkpoints"/f"{mode}_seed{seed}_fold{fold}.pt"
            if device.type=="cuda": torch.cuda.reset_peak_memory_stats()
            train_start=time.perf_counter()
            for ep in range(1,max_epochs+1):
                tr=epoch_pass(model,train_dl,criterion,device,optimizer,scaler)
                va=epoch_pass(model,val_dl,criterion,device)
                for split,res in [("train",tr),("inner_validation",va)]:
                    epochs_out.append({"mode":mode,"seed":seed,"outer_fold":fold,
                                       "epoch":ep,"split":split,**res})
                if va["macro_f1"] > best + 1e-4:
                    best=va["macro_f1"]; best_epoch=ep; stale=0
                    torch.save({"model":model.state_dict(),"epoch":ep,"best_macro_f1":best},ckpt)
                else: stale+=1
                print(
                    f"seed={seed} mode={mode} fold={fold} epoch={ep}/{max_epochs} "
                    f"train_loss={tr['loss']:.4f} val_macroF1={va['macro_f1']:.4f} "
                    f"train_s={tr['seconds']:.1f} val_s={va['seconds']:.1f}",
                    flush=True
                )
                if stale>=CONFIG["PATIENCE"]: break
            training_seconds=time.perf_counter()-train_start
            model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True)["model"])
            p, infer_seconds=predict(model,test_dl,device)
            p=p.merge(outer_test[["sample_id","class_label"]].rename(columns={"class_label":"true_label"}),
                      on="sample_id",validate="one_to_one")
            p["mode"]=mode; p["seed"]=seed
            p["correct"]=(p.true_label==p.predicted_label).astype(int)
            rep=classification_report(p.true_label,p.predicted_label,labels=labels,
                                      output_dict=True,zero_division=0)
            total_params=sum(x.numel() for x in model.parameters())
            trainable=sum(x.numel() for x in model.parameters() if x.requires_grad)
            peak=(torch.cuda.max_memory_allocated()/1024**2) if device.type=="cuda" else 0
            folds_out.append({
                "mode":mode,"seed":seed,"outer_fold":fold,"best_epoch":best_epoch,
                "best_inner_macro_f1":best,
                "accuracy":accuracy_score(p.true_label,p.predicted_label),
                "macro_f1":f1_score(p.true_label,p.predicted_label,average="macro",zero_division=0),
                "balanced_accuracy":balanced_accuracy_score(p.true_label,p.predicted_label),
                "training_seconds":training_seconds,
                "test_inference_seconds":infer_seconds,
                "milliseconds_per_test_image":1000*infer_seconds/len(p),
                "peak_vram_mb":peak,"total_parameters":total_params,
                "trainable_parameters":trainable,
                **{f"recall_{c}":rep[c]["recall"] for c in labels}
            })
            preds_out.extend(p.to_dict("records"))
            ckpt_out.append({"mode":mode,"seed":seed,"outer_fold":fold,
                             "checkpoint_path":str(ckpt),
                             "checkpoint_size_mb":ckpt.stat().st_size/1024**2,
                             "best_epoch":best_epoch,"trainable_parameters":trainable})
            del model,optimizer,train_dl,val_dl,test_dl
            gc.collect()
            if device.type=="cuda": torch.cuda.empty_cache()

epoch_df=pd.DataFrame(epochs_out)
fold_df=pd.DataFrame(folds_out)
ft_pred=pd.DataFrame(preds_out)
ckpt_df=pd.DataFrame(ckpt_out)

epoch_df.to_csv(CONFIG["OUTPUT_ROOT"]/"transformer_epoch_history.csv",index=False)
fold_df.to_csv(CONFIG["OUTPUT_ROOT"]/"transformer_outer_fold_metrics.csv",index=False)
ft_pred.to_csv(CONFIG["OUTPUT_ROOT"]/"transformer_sample_predictions.csv",index=False)
ckpt_df.to_csv(CONFIG["OUTPUT_ROOT"]/"transformer_checkpoint_summary.csv",index=False)

expected=len(CONFIG["SEEDS"])*len(CONFIG["TRAINING_MODES"])*594
if len(ft_pred)!=expected or ft_pred.duplicated(["mode","seed","sample_id"]).any():
    raise RuntimeError("Fine-tuned prediction coverage validation failed.")

nested=pd.read_csv(CONFIG["NESTED_PREDICTIONS_FILE"],dtype={"sample_id":str})
mapping={"B_VIT_B16_768D":"FROZEN_VIT_SVM",
         "E_FUSION_PCA128_PROPOSED":"PROPOSED_HYBRID"}
base=nested[nested.scenario.isin(mapping)].copy()
base["mode"]=base.scenario.map(mapping)
cols=["sample_id","source_dataset","outer_fold","true_label",
      "predicted_label","correct","seed","mode"]
comparison=pd.concat([base[cols],ft_pred[cols]],ignore_index=True)
comparison.to_csv(CONFIG["OUTPUT_ROOT"]/"transformer_comparison_predictions.csv",index=False)

seed_rows=[]
for (mode,seed),g in comparison.groupby(["mode","seed"]):
    seed_rows.append({"mode":mode,"seed":seed,"n":len(g),
                      "accuracy":accuracy_score(g.true_label,g.predicted_label),
                      "macro_f1":f1_score(g.true_label,g.predicted_label,average="macro",zero_division=0),
                      "balanced_accuracy":balanced_accuracy_score(g.true_label,g.predicted_label)})
seed_df=pd.DataFrame(seed_rows)
seed_df.to_csv(CONFIG["OUTPUT_ROOT"]/"transformer_seed_level_metrics.csv",index=False)
summary=seed_df.groupby("mode",as_index=False).agg(
    n_seeds=("seed","nunique"),mean_accuracy=("accuracy","mean"),sd_accuracy=("accuracy","std"),
    mean_macro_f1=("macro_f1","mean"),sd_macro_f1=("macro_f1","std"),
    mean_balanced_accuracy=("balanced_accuracy","mean"),sd_balanced_accuracy=("balanced_accuracy","std"))
summary.to_csv(CONFIG["OUTPUT_ROOT"]/"transformer_multi_seed_summary.csv",index=False)

paired_dir=CONFIG["OUTPUT_ROOT"]/"paired_matrices_by_seed"
paired_dir.mkdir(exist_ok=True)
modes=["FROZEN_VIT_SVM","PROPOSED_HYBRID","PARTIAL_LAST_BLOCK","FULL_FINE_TUNING"]
for seed in CONFIG["SEEDS"]:
    x=comparison[comparison.seed==seed]
    paired=x[["sample_id","source_dataset","outer_fold","true_label"]].drop_duplicates().sort_values("sample_id")
    for mode in modes:
        z=x.loc[x["mode"] == mode][["sample_id","predicted_label","correct"]].rename(
            columns={"predicted_label":f"pred_{mode}","correct":f"correct_{mode}"})
        paired=paired.merge(z,on="sample_id",validate="one_to_one")
    paired.to_csv(paired_dir/f"transformer_paired_matrix_seed_{seed}.csv",index=False)

comp=fold_df.groupby("mode",as_index=False).agg(
    mean_training_seconds=("training_seconds","mean"),
    sd_training_seconds=("training_seconds","std"),
    mean_inference_ms_per_image=("milliseconds_per_test_image","mean"),
    mean_peak_vram_mb=("peak_vram_mb","mean"),
    mean_best_epoch=("best_epoch","mean"),
    trainable_parameters=("trainable_parameters","mean"))
comp.to_csv(CONFIG["OUTPUT_ROOT"]/"transformer_computational_summary.csv",index=False)

metadata={
    "analysis_name":"AMD Transformer Comparison Reviewer v1",
    "model_name":CONFIG["MODEL_NAME"],"n_samples":594,
    "training_modes":CONFIG["TRAINING_MODES"],"seeds":CONFIG["SEEDS"],
    "outer_folds":[1,2,3,4,5],"validation_fraction":CONFIG["VALIDATION_FRACTION"],
    "max_epochs":CONFIG["MAX_EPOCHS"],"patience":CONFIG["PATIENCE"],
    "batch_size":CONFIG["BATCH_SIZE"],"learning_rate":CONFIG["LEARNING_RATE"],
    "outer_fold_sha256":sha256(CONFIG["OUTER_FOLD_FILE"]),
    "torch_version":torch.__version__,"timm_version":timm.__version__,
    "device":str(device),
    "notes":[
        "No ROI was manually recropped.",
        "All methods use the same frozen ROI images and immutable outer folds.",
        "Frozen ViT-SVM and proposed predictions are imported from the audited nested-CV experiment.",
        "Partial fine-tuning trains the final transformer block, final norm, and classifier head.",
        "Full fine-tuning updates all ViT-B/16 parameters.",
        "Early stopping uses only an inner validation split from each outer-training partition."
    ]}
(CONFIG["OUTPUT_ROOT"]/"run_metadata.json").write_text(json.dumps(metadata,indent=2))
(CONFIG["OUTPUT_ROOT"]/"AUDIT_REPORT.md").write_text(
"""# Transformer Comparison Audit

Four strategies are compared on the same 594 frozen ROI images and immutable
outer folds: frozen ViT-SVM, proposed hybrid PCA-SVM, partial fine-tuning, and
full fine-tuning. No ROI is manually cropped again. The outer test fold is never
used for checkpoint selection. Fine-tuned models use an inner validation split
from the outer-training data, and one out-of-fold prediction is saved for every
sample, seed, and method.
""")
print(summary)
print("Outputs:",CONFIG["OUTPUT_ROOT"].resolve())
